# Corporate Signal Intelligence  
## Feature Engineering Notebook

> This notebook transforms the cleaned market, corporate filing, and financial fundamentals datasets into model-ready analytical features for anomaly detection, corporate risk scoring, and executive intelligence.

---

## Project Stage

The previous notebook completed the data analysis and cleansing stage, producing four validated datasets:

```text
clean_market_data
clean_company_metadata
clean_sec_filings
clean_sec_facts
```

This notebook now focuses on transforming those cleaned datasets into structured features that can be consumed by machine learning models, REST API endpoints, and dashboard visualizations.

---

## Feature Engineering Objective

The goal of this stage is to create meaningful signals from three main data layers:

| Layer | Dataset | Feature Type |
|---|---|---|
| **Market Data** | `clean_market_data` | Price, volume, return, volatility, anomaly signals |
| **SEC Filings** | `clean_sec_filings` | Filing activity, event frequency, disclosure timing |
| **SEC Financial Facts** | `clean_sec_facts` | Financial ratios, growth indicators, corporate fundamentals |
| **Company Metadata** | `clean_company_metadata` | Ticker, CIK, company identity enrichment |

The final output should be a set of model-ready datasets that can support both machine learning and business interpretation.

---

## Planned Feature Groups

### Market Features

The market data layer will be used to create features such as:

```text
daily_return
price_change_7d
price_change_30d
volatility_7d
volatility_30d
volume_change_30d
volume_zscore_30d
return_zscore_30d
```

These features will support the first anomaly detection models using Scikit-learn.

---

### Filing Activity Features

The SEC filings layer will be used to create corporate disclosure features such as:

```text
filing_count_30d
filing_count_90d
recent_10k_count
recent_10q_count
recent_8k_count
days_since_last_filing
days_since_last_8k
```

These features will help detect unusual corporate reporting activity and recent material event signals.

---

### Financial Fundamental Features

The SEC company facts layer will be used to generate financial ratios and growth indicators such as:

```text
revenue_growth
net_income_growth
operating_margin
liabilities_to_assets
cash_to_assets
equity_to_assets
rd_to_revenue
```

These variables will help represent corporate fundamentals in a structured and model-ready format.

---

## Expected Outputs

By the end of this notebook, the project should produce feature datasets such as:

```text
market_features_df
filing_features_df
financial_features_df
model_ready_df
```

The `model_ready_df` will serve as the main analytical dataset for the next stage of the project: anomaly detection and corporate risk scoring.

---

## Final Pipeline Context

```text
Clean Market Data
        +
Clean SEC Filings
        +
Clean SEC Financial Facts
        ↓
Feature Engineering
        ↓
Model-Ready Dataset
        ↓
Scikit-learn Anomaly Detection
        ↓
Groq Executive Briefings
        ↓
FastAPI + Dashboard Layer
```

This notebook represents the bridge between clean data and the machine learning layer of the Corporate Signal Intelligence platform.

In [30]:
# Install required Python packages

%pip install pandas numpy scipy scikit-learn matplotlib seaborn plotly pyarrow

print("\nDependencies installed successfully!")

Note: you may need to restart the kernel to use updated packages.

Dependencies installed successfully!


In [31]:
# Analysing the enviroment

import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: /home/a1rm4x/Documents/GitHub/corporate-signal-intelligence/.venv/bin/python
Python version: 3.14.4 (main, Apr  8 2026, 17:48:49) [GCC 15.2.1 20260209]


In [32]:
# Importing libraries

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from scipy import stats
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DATA_DIR = Path("data")

print("Libraries loaded successfully.")
print("Data directory:", DATA_DIR.resolve())

Libraries loaded successfully.
Data directory: /home/a1rm4x/Documents/GitHub/corporate-signal-intelligence/data


In [33]:
# Loading the clean datasets

clean_market_df = pd.read_csv(DATA_DIR / "clean_market_data.csv")
clean_companies_df = pd.read_csv(DATA_DIR / "clean_company_metadata.csv")
clean_sec_filings_df = pd.read_csv(DATA_DIR / "clean_sec_filings.csv")
clean_sec_facts_df = pd.read_csv(DATA_DIR / "clean_sec_facts.csv")

print("clean_market_df:", clean_market_df.shape)
print("clean_companies_df:", clean_companies_df.shape)
print("clean_sec_filings_df:", clean_sec_filings_df.shape)
print("clean_sec_facts_df:", clean_sec_facts_df.shape)

clean_market_df: (28630, 9)
clean_companies_df: (10, 5)
clean_sec_filings_df: (1092, 10)
clean_sec_facts_df: (14797, 18)


In [34]:
# Converting the data columns to datetime

clean_market_df["date"] = pd.to_datetime(clean_market_df["date"], errors="coerce")
clean_market_df["collected_at"] = pd.to_datetime(clean_market_df["collected_at"], errors="coerce")

clean_companies_df["collected_at"] = pd.to_datetime(clean_companies_df["collected_at"], errors="coerce")

clean_sec_filings_df["filing_date"] = pd.to_datetime(clean_sec_filings_df["filing_date"], errors="coerce")
clean_sec_filings_df["report_date"] = pd.to_datetime(clean_sec_filings_df["report_date"], errors="coerce")
clean_sec_filings_df["collected_at"] = pd.to_datetime(clean_sec_filings_df["collected_at"], errors="coerce")

date_columns = ["start_date", "end_date", "filing_date", "reference_date", "collected_at"]

for col in date_columns:
    if col in clean_sec_facts_df.columns:
        clean_sec_facts_df[col] = pd.to_datetime(clean_sec_facts_df[col], errors="coerce")

print("Clean datasets loaded and date columns converted.")

Clean datasets loaded and date columns converted.


In [35]:
# Market Feature Engineering


def build_market_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build market-based features from daily OHLCV data.

    Features include:
    - daily returns
    - price changes over 7 and 30 trading days
    - rolling volatility
    - volume change against rolling average
    - return and volume z-scores
    """
    feature_frames = []

    for ticker, group in df.groupby("ticker"):
        group = group.copy()
        group = group.sort_values("date")

        # Returns
        group["daily_return"] = group["close"].pct_change()
        group["log_return"] = np.log(group["close"] / group["close"].shift(1))

        # Price momentum
        group["price_change_7d"] = group["close"].pct_change(periods=7)
        group["price_change_30d"] = group["close"].pct_change(periods=30)
        group["price_change_90d"] = group["close"].pct_change(periods=90)

        # Rolling volatility
        group["volatility_7d"] = group["daily_return"].rolling(window=7).std()
        group["volatility_30d"] = group["daily_return"].rolling(window=30).std()
        group["volatility_90d"] = group["daily_return"].rolling(window=90).std()

        # Volume rolling statistics
        group["avg_volume_30d"] = group["volume"].rolling(window=30).mean()
        group["std_volume_30d"] = group["volume"].rolling(window=30).std()

        # Volume deviation from recent baseline
        group["volume_change_30d"] = (
            group["volume"] - group["avg_volume_30d"]
        ) / group["avg_volume_30d"]

        group["volume_zscore_30d"] = (
            group["volume"] - group["avg_volume_30d"]
        ) / group["std_volume_30d"]

        # Return rolling z-score
        group["avg_return_30d"] = group["daily_return"].rolling(window=30).mean()
        group["std_return_30d"] = group["daily_return"].rolling(window=30).std()

        group["return_zscore_30d"] = (
            group["daily_return"] - group["avg_return_30d"]
        ) / group["std_return_30d"]

        # Intraday range
        group["daily_range"] = (group["high"] - group["low"]) / group["close"]

        # Gap between current open and previous close
        group["open_gap"] = (group["open"] - group["close"].shift(1)) / group["close"].shift(1)

        feature_frames.append(group)

    market_features_df = pd.concat(feature_frames, ignore_index=True)

    # Remove rows without enough rolling window history
    market_features_df = market_features_df.dropna().reset_index(drop=True)

    return market_features_df


market_features_df = build_market_features(clean_market_df)

print("market_features_df shape:", market_features_df.shape)

market_features_df.head()

market_features_df shape: (27730, 26)


,ticker,date,open,high,low,close,volume,source,collected_at,daily_return,log_return,price_change_7d,price_change_30d,price_change_90d,volatility_7d,volatility_30d,volatility_90d,avg_volume_30d,std_volume_30d,volume_change_30d,volume_zscore_30d,avg_return_30d,std_return_30d,return_zscore_30d,daily_range,open_gap
0,AAPL,2015-05-13,28.1670,28.4023,28.1062,28.1376,"155,379,331.0000",stooq,2026-05-21 19:51:02.287737+00:00,0.0012,0.0012,-0.0167,0.0170,0.1620,0.0130,0.0132,0.0161,"217,353,230.9000","94,016,158.6912",-0.2851,-0.6592,0.0006,0.0132,0.0411,0.0105,0.0022
1,AAPL,2015-05-14,28.4505,28.7970,28.3955,28.7970,"202,444,982.0000",stooq,2026-05-21 19:51:02.287737+00:00,0.0234,0.0232,0.0293,0.0423,0.2237,0.0128,0.0138,0.0159,"218,011,922.3333","93,833,933.5375",-0.0714,-0.1659,0.0015,0.0138,1.5892,0.0139,0.0111
2,AAPL,2015-05-15,28.8244,28.9237,28.6280,28.7557,"171,052,227.0000",stooq,2026-05-21 19:51:02.287737+00:00,-0.0014,-0.0014,0.0345,0.0320,0.2218,0.0122,0.0138,0.0159,"218,883,610.9333","93,250,996.0012",-0.2185,-0.5129,0.0011,0.0138,-0.1871,0.0103,0.0010
3,AAPL,2015-05-18,28.6644,29.1845,28.6596,29.0639,"227,617,764.0000",stooq,2026-05-21 19:51:02.287737+00:00,0.0107,0.0107,0.0390,0.0265,0.2176,0.0124,0.0136,0.0159,"220,895,204.6667","92,748,755.8727",0.0304,0.0725,0.0010,0.0136,0.7179,0.0181,-0.0032
4,AAPL,2015-05-19,29.1776,29.2201,28.9530,29.0373,"199,891,497.0000",stooq,2026-05-21 19:51:02.287737+00:00,-0.0009,-0.0009,0.0190,0.0364,0.1716,0.0111,0.0134,0.0154,"222,312,705.3667","92,066,804.2472",-0.1009,-0.2435,0.0013,0.0134,-0.1633,0.0092,0.0039


In [36]:
# Validating coverage

market_features_df.groupby("ticker").agg(
    rows=("date", "count"),
    min_date=("date", "min"),
    max_date=("date", "max"),
    avg_daily_return=("daily_return", "mean"),
    avg_volatility_30d=("volatility_30d", "mean"),
    max_volume_zscore=("volume_zscore_30d", "max"),
    min_return_zscore=("return_zscore_30d", "min"),
).sort_values("ticker")

,rows,min_date,max_date,avg_daily_return,avg_volatility_30d,max_volume_zscore,min_return_zscore
ticker,,,,,,,
AAPL,2773,2015-05-13,2026-05-21,0.0010,0.0166,5.2031,-3.7959
AMD,2773,2015-05-13,2026-05-21,0.0026,0.0351,5.0834,-4.6285
AMZN,2773,2015-05-13,2026-05-21,0.0011,0.0192,5.0465,-4.1858
GOOGL,2773,2015-05-13,2026-05-21,0.0011,0.0171,4.8367,-4.4354
INTC,2773,2015-05-13,2026-05-21,0.0009,0.0230,5.0393,-4.9117
META,2773,2015-05-13,2026-05-21,0.0010,0.0216,5.0611,-4.9281
MSFT,2773,2015-05-13,2026-05-21,0.0010,0.0156,4.9207,-4.3544
NVDA,2773,2015-05-13,2026-05-21,0.0027,0.0287,5.0771,-4.0238
ORCL,2773,2015-05-13,2026-05-21,0.0008,0.0181,5.1268,-4.7756


In [37]:
# Saving the first version of the market features dataset

market_features_df.to_csv("data/market_features.csv", index=False)

print("Market features saved successfully.")

Market features saved successfully.


## Market Feature Engineering — Initial Results

The market feature engineering stage was successfully completed using the cleaned Stooq market dataset.

The generated `market_features_df` contains a standardized feature set for all monitored tickers, covering the period from 2015-05-13 to 2026-05-21. Each ticker contains the same number of usable observations after applying rolling windows, which creates a balanced dataset for the anomaly detection stage.

The generated features include daily returns, log returns, price momentum over multiple windows, rolling volatility, volume deviation, return z-scores, intraday price range, and opening price gaps.

These features will be used as the first model-ready input layer for market anomaly detection.

Key feature groups:

```text
daily_return
log_return
price_change_7d
price_change_30d
price_change_90d
volatility_7d
volatility_30d
volatility_90d
volume_change_30d
volume_zscore_30d
return_zscore_30d
daily_range
open_gap

In [38]:
# SEC Filing Feature Engineering

def build_filing_features(
    filings_df: pd.DataFrame,
    market_dates_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build daily SEC filing activity features aligned with market dates.

    The output will have one row per ticker/date from the market dataset,
    enriched with rolling filing activity indicators.
    """
    filings = filings_df.copy()
    market_dates = market_dates_df[["ticker", "date"]].drop_duplicates().copy()

    filings["filing_date"] = pd.to_datetime(filings["filing_date"], errors="coerce")
    market_dates["date"] = pd.to_datetime(market_dates["date"], errors="coerce")

    filings["ticker"] = filings["ticker"].str.upper().str.strip()
    market_dates["ticker"] = market_dates["ticker"].str.upper().str.strip()

    # Keep only valid filing dates
    filings = filings.dropna(subset=["ticker", "filing_date", "form_type"]).copy()

    # Normalize filing date to date only
    filings["filing_date"] = filings["filing_date"].dt.normalize()
    market_dates["date"] = market_dates["date"].dt.normalize()

    feature_frames = []

    for ticker, market_group in market_dates.groupby("ticker"):
        market_group = market_group.copy().sort_values("date")

        ticker_filings = filings[filings["ticker"] == ticker].copy()

        if ticker_filings.empty:
            market_group["filing_count"] = 0
            market_group["form_10k_count"] = 0
            market_group["form_10q_count"] = 0
            market_group["form_8k_count"] = 0
        else:
            daily_counts = (
                ticker_filings
                .groupby(["filing_date", "form_type"])
                .size()
                .reset_index(name="count")
            )

            pivot_counts = daily_counts.pivot_table(
                index="filing_date",
                columns="form_type",
                values="count",
                aggfunc="sum",
                fill_value=0,
            ).reset_index()

            pivot_counts = pivot_counts.rename(columns={"filing_date": "date"})

            market_group = market_group.merge(
                pivot_counts,
                on="date",
                how="left",
            )

            # Create safe columns for important forms
            market_group["form_10k_count"] = market_group.get("10-K", 0)
            market_group["form_10q_count"] = market_group.get("10-Q", 0)
            market_group["form_8k_count"] = market_group.get("8-K", 0)

            form_columns = [
                col for col in ["10-K", "10-Q", "8-K"]
                if col in market_group.columns
            ]

            if form_columns:
                market_group["filing_count"] = market_group[form_columns].sum(axis=1)
            else:
                market_group["filing_count"] = 0

        # Fill missing filing counts
        count_columns = [
            "filing_count",
            "form_10k_count",
            "form_10q_count",
            "form_8k_count",
        ]

        for col in count_columns:
            if col not in market_group.columns:
                market_group[col] = 0

            market_group[col] = market_group[col].fillna(0).astype(int)

        # Rolling filing activity
        market_group["filing_count_30d"] = (
            market_group["filing_count"]
            .rolling(window=30, min_periods=1)
            .sum()
        )

        market_group["filing_count_90d"] = (
            market_group["filing_count"]
            .rolling(window=90, min_periods=1)
            .sum()
        )

        market_group["form_8k_count_30d"] = (
            market_group["form_8k_count"]
            .rolling(window=30, min_periods=1)
            .sum()
        )

        market_group["form_10q_count_180d"] = (
            market_group["form_10q_count"]
            .rolling(window=180, min_periods=1)
            .sum()
        )

        market_group["form_10k_count_365d"] = (
            market_group["form_10k_count"]
            .rolling(window=365, min_periods=1)
            .sum()
        )

        # Days since last filing
        last_filing_date = None
        days_since_last_filing = []
        days_since_last_8k = []
        last_8k_date = None

        for _, row in market_group.iterrows():
            current_date = row["date"]

            if row["filing_count"] > 0:
                last_filing_date = current_date

            if row["form_8k_count"] > 0:
                last_8k_date = current_date

            if last_filing_date is None:
                days_since_last_filing.append(np.nan)
            else:
                days_since_last_filing.append((current_date - last_filing_date).days)

            if last_8k_date is None:
                days_since_last_8k.append(np.nan)
            else:
                days_since_last_8k.append((current_date - last_8k_date).days)

        market_group["days_since_last_filing"] = days_since_last_filing
        market_group["days_since_last_8k"] = days_since_last_8k

        feature_frames.append(market_group)

    filing_features_df = pd.concat(feature_frames, ignore_index=True)

    # Fill early-period missing values with a large number to indicate no recent known filing
    filing_features_df["days_since_last_filing"] = (
        filing_features_df["days_since_last_filing"].fillna(9999)
    )

    filing_features_df["days_since_last_8k"] = (
        filing_features_df["days_since_last_8k"].fillna(9999)
    )

    # Keep only relevant feature columns
    filing_features_df = filing_features_df[
        [
            "ticker",
            "date",
            "filing_count",
            "form_10k_count",
            "form_10q_count",
            "form_8k_count",
            "filing_count_30d",
            "filing_count_90d",
            "form_8k_count_30d",
            "form_10q_count_180d",
            "form_10k_count_365d",
            "days_since_last_filing",
            "days_since_last_8k",
        ]
    ]

    return filing_features_df.sort_values(["ticker", "date"]).reset_index(drop=True)


filing_features_df = build_filing_features(
    filings_df=clean_sec_filings_df,
    market_dates_df=market_features_df,
)

print("filing_features_df shape:", filing_features_df.shape)

filing_features_df.head()

filing_features_df shape: (27730, 13)


,ticker,date,filing_count,form_10k_count,form_10q_count,form_8k_count,filing_count_30d,filing_count_90d,form_8k_count_30d,form_10q_count_180d,form_10k_count_365d,days_since_last_filing,days_since_last_8k
0,AAPL,2015-05-13,1,0,0,1,1.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000
1,AAPL,2015-05-14,0,0,0,0,1.0000,1.0000,1.0000,0.0000,0.0000,1.0000,1.0000
2,AAPL,2015-05-15,0,0,0,0,1.0000,1.0000,1.0000,0.0000,0.0000,2.0000,2.0000
3,AAPL,2015-05-18,0,0,0,0,1.0000,1.0000,1.0000,0.0000,0.0000,5.0000,5.0000
4,AAPL,2015-05-19,0,0,0,0,1.0000,1.0000,1.0000,0.0000,0.0000,6.0000,6.0000


In [39]:
# Verifying coverage

filing_features_df.groupby("ticker").agg(
    rows=("date", "count"),
    min_date=("date", "min"),
    max_date=("date", "max"),
    total_filings=("filing_count", "sum"),
    total_10k=("form_10k_count", "sum"),
    total_10q=("form_10q_count", "sum"),
    total_8k=("form_8k_count", "sum"),
    max_filing_count_30d=("filing_count_30d", "max"),
    min_days_since_last_filing=("days_since_last_filing", "min"),
).sort_values("ticker")

,rows,min_date,max_date,total_filings,total_10k,total_10q,total_8k,max_filing_count_30d,min_days_since_last_filing
ticker,,,,,,,,,
AAPL,2773,2015-05-13,2026-05-21,149,11,33,105,5.0000,0.0000
AMD,2773,2015-05-13,2026-05-21,145,9,27,109,6.0000,0.0000
AMZN,2773,2015-05-13,2026-05-21,88,6,19,63,5.0000,0.0000
GOOGL,2773,2015-05-13,2026-05-21,52,4,10,38,5.0000,0.0000
INTC,2773,2015-05-13,2026-05-21,149,7,22,120,8.0000,0.0000
META,2773,2015-05-13,2026-05-21,31,2,7,22,4.0000,0.0000
MSFT,2773,2015-05-13,2026-05-21,92,6,20,66,7.0000,0.0000
NVDA,2773,2015-05-13,2026-05-21,87,6,19,62,6.0000,0.0000
ORCL,2773,2015-05-13,2026-05-21,137,10,32,95,5.0000,0.0000


In [40]:
# Checking for null values

filing_features_df.isna().sum().sort_values(ascending=False)

ticker                    0
date                      0
filing_count              0
form_10k_count            0
form_10q_count            0
form_8k_count             0
filing_count_30d          0
filing_count_90d          0
form_8k_count_30d         0
form_10q_count_180d       0
form_10k_count_365d       0
days_since_last_filing    0
days_since_last_8k        0
dtype: int64

In [41]:
# Saving the new dataset to disk

filing_features_df.to_csv("data/filing_features.csv", index=False)

print("Filing features saved successfully.")

Filing features saved successfully.


In [42]:
# Checking what conceps we have bty ticker

clean_sec_facts_df.groupby(["ticker", "concept"]).agg(
    rows=("value", "count"),
    min_date=("reference_date", "min"),
    max_date=("reference_date", "max"),
).reset_index().sort_values(["ticker", "concept"])

,ticker,concept,rows,min_date,max_date
0,AAPL,Assets,144,2008-09-27,2026-03-28
1,AAPL,CashAndCashEquivalentsAtCarryingValue,226,2006-09-30,2026-03-28
2,AAPL,Liabilities,142,2008-09-27,2026-03-28
3,AAPL,NetIncomeLoss,334,2007-09-29,2026-03-28
4,AAPL,OperatingIncomeLoss,230,2007-09-29,2026-03-28
5,AAPL,ResearchAndDevelopmentExpense,230,2007-09-29,2026-03-28
6,AAPL,RevenueFromContractWithCustomerExcludingAssess...,113,2017-09-30,2026-03-28
7,AAPL,Revenues,11,2016-09-24,2018-09-29
8,AAPL,StockholdersEquity,258,2006-09-30,2026-03-28
9,AMD,Assets,128,2009-12-26,2026-03-28


In [ ]:
# Financial Feature Engineering - v2

def build_financial_features_v2(facts_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build cleaner financial features from SEC XBRL company facts.

    This function:
    - maps equivalent SEC concepts into unified feature names
    - filters USD records
    - keeps 10-K and 10-Q reports
    - deduplicates repeated SEC observations
    - pivots concepts into columns
    - forward-fills reported financial values by ticker
    - creates financial ratios and growth indicators
    """

    df = facts_df.copy()

    # Basic standardization
    df["ticker"] = df["ticker"].str.upper().str.strip()
    df["concept"] = df["concept"].str.strip()
    df["unit"] = df["unit"].str.upper().str.strip()
    df["form_type"] = df["form_type"].str.upper().str.strip()

    df["reference_date"] = pd.to_datetime(df["reference_date"], errors="coerce")
    df["filing_date"] = pd.to_datetime(df["filing_date"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    # Keep only model-relevant records
    df = df[
        (df["unit"] == "USD")
        & (df["form_type"].isin(["10-K", "10-Q"]))
        & (df["reference_date"].notna())
        & (df["value"].notna())
    ].copy()

    # Map SEC concepts to unified feature names
    concept_map = {
        "RevenueFromContractWithCustomerExcludingAssessedTax": "revenue",
        "Revenues": "revenue",
        "NetIncomeLoss": "net_income",
        "OperatingIncomeLoss": "operating_income",
        "Assets": "assets",
        "Liabilities": "liabilities",
        "StockholdersEquity": "stockholders_equity",
        "CashAndCashEquivalentsAtCarryingValue": "cash_and_equivalents",
        "ResearchAndDevelopmentExpense": "rd_expense",
    }

    df = df[df["concept"].isin(concept_map.keys())].copy()
    df["feature_name"] = df["concept"].map(concept_map)

    # Keep the latest filing when the same metric/date appears multiple times
    df = df.sort_values(
        ["ticker", "feature_name", "reference_date", "filing_date"],
        ascending=[True, True, True, False],
    )

    df = df.drop_duplicates(
        subset=[
            "ticker",
            "feature_name",
            "reference_date",
            "fiscal_year",
            "fiscal_period",
        ],
        keep="first",
    )

    # Pivot long SEC facts into wide financial features
    financial_wide = df.pivot_table(
        index=["ticker", "reference_date", "fiscal_year", "fiscal_period"],
        columns="feature_name",
        values="value",
        aggfunc="last",
    ).reset_index()

    financial_wide.columns.name = None

    expected_cols = [
        "revenue",
        "net_income",
        "operating_income",
        "assets",
        "liabilities",
        "stockholders_equity",
        "cash_and_equivalents",
        "rd_expense",
    ]

    for col in expected_cols:
        if col not in financial_wide.columns:
            financial_wide[col] = np.nan

    financial_wide = financial_wide.sort_values(
        ["ticker", "reference_date"]
    ).reset_index(drop=True)

    # Forward-fill reported values within each ticker
    financial_wide[expected_cols] = (
        financial_wide.groupby("ticker")[expected_cols].ffill()
    )

    # Financial ratios
    financial_wide["operating_margin"] = (
        financial_wide["operating_income"] / financial_wide["revenue"]
    )

    financial_wide["net_margin"] = (
        financial_wide["net_income"] / financial_wide["revenue"]
    )

    financial_wide["liabilities_to_assets"] = (
        financial_wide["liabilities"] / financial_wide["assets"]
    )

    financial_wide["cash_to_assets"] = (
        financial_wide["cash_and_equivalents"] / financial_wide["assets"]
    )

    financial_wide["equity_to_assets"] = (
        financial_wide["stockholders_equity"] / financial_wide["assets"]
    )

    financial_wide["rd_to_revenue"] = (
        financial_wide["rd_expense"] / financial_wide["revenue"]
    )

    # Growth features by ticker
    frames = []

    for ticker, group in financial_wide.groupby("ticker"):
        group = group.copy().sort_values("reference_date")

        group["revenue_growth_qoq"] = group["revenue"].pct_change(1)
        group["revenue_growth_yoy"] = group["revenue"].pct_change(4)

        group["net_income_growth_qoq"] = group["net_income"].pct_change(1)
        group["net_income_growth_yoy"] = group["net_income"].pct_change(4)

        group["assets_growth_qoq"] = group["assets"].pct_change(1)
        group["liabilities_growth_qoq"] = group["liabilities"].pct_change(1)

        frames.append(group)

    financial_features_df = pd.concat(frames, ignore_index=True)

    # Replace invalid divisions
    financial_features_df = financial_features_df.replace([np.inf, -np.inf], np.nan)

    # Align with market model period
    financial_features_df = financial_features_df[
        financial_features_df["reference_date"] >= "2015-01-01"
    ].copy()

    financial_features_df = financial_features_df.sort_values(
        ["ticker", "reference_date"]
    ).reset_index(drop=True)

    return financial_features_df

In [47]:
# Consolidating coverage

financial_features_df = build_financial_features_v2(clean_sec_facts_df)

print("financial_features_df shape:", financial_features_df.shape)
display(financial_features_df.head())

financial_features_df shape: (1904, 24)


,ticker,reference_date,fiscal_year,fiscal_period,assets,cash_and_equivalents,liabilities,net_income,operating_income,rd_expense,revenue,stockholders_equity,operating_margin,net_margin,liabilities_to_assets,cash_to_assets,equity_to_assets,rd_to_revenue,revenue_growth_qoq,revenue_growth_yoy,net_income_growth_qoq,net_income_growth_yoy,assets_growth_qoq,liabilities_growth_qoq
0,AAPL,2015-03-28,"2,016.0000",Q2,"261,194,000,000.0000","14,489,000,000.0000","132,188,000,000.0000","31,593,000,000.0000","42,524,000,000.0000","3,813,000,000.0000",NaN,"129,006,000,000.0000",NaN,NaN,0.5061,0.0555,0.4939,NaN,NaN,NaN,0.7528,0.7528,-0.0027,-0.0460
1,AAPL,2015-03-28,"2,016.0000",FY,"261,194,000,000.0000","14,489,000,000.0000","132,188,000,000.0000","13,569,000,000.0000","42,524,000,000.0000","3,813,000,000.0000",NaN,"129,006,000,000.0000",NaN,NaN,0.5061,0.0555,0.4939,NaN,NaN,NaN,-0.5705,-0.2472,0.0000,0.0000
2,AAPL,2015-03-28,"2,015.0000",Q2,"261,194,000,000.0000","14,489,000,000.0000","132,188,000,000.0000","31,593,000,000.0000","42,524,000,000.0000","3,813,000,000.0000",NaN,"129,006,000,000.0000",NaN,NaN,0.5061,0.0555,0.4939,NaN,NaN,NaN,1.3283,0.7528,0.0000,0.0000
3,AAPL,2015-03-28,"2,015.0000",FY,"261,894,000,000.0000","19,478,000,000.0000","138,566,000,000.0000","13,569,000,000.0000","24,246,000,000.0000","1,895,000,000.0000",NaN,"123,328,000,000.0000",NaN,NaN,0.5291,0.0744,0.4709,NaN,NaN,NaN,-0.5705,-0.2472,0.0027,0.0482
4,AAPL,2015-06-27,"2,015.0000",FY,"261,194,000,000.0000","14,489,000,000.0000","132,188,000,000.0000","10,677,000,000.0000","42,524,000,000.0000","3,813,000,000.0000",NaN,"129,006,000,000.0000",NaN,NaN,0.5061,0.0555,0.4939,NaN,NaN,NaN,-0.2131,-0.6620,-0.0027,-0.0460


In [48]:
# Analysing missing values

missing_values = financial_features_df.isna().sum()
missing_pct = financial_features_df.isna().mean()

financial_missing_summary = pd.DataFrame({
    "column": missing_values.index,
    "missing_values": missing_values.values,
    "missing_pct": missing_pct.values,
})

financial_missing_summary = financial_missing_summary.sort_values(
    "missing_pct",
    ascending=False
).reset_index(drop=True)

financial_missing_summary

,column,missing_values,missing_pct
0,liabilities_growth_qoq,708,0.3718
1,liabilities,705,0.3703
2,liabilities_to_assets,705,0.3703
3,rd_to_revenue,334,0.1754
4,rd_expense,233,0.1224
5,revenue_growth_yoy,172,0.0903
6,revenue_growth_qoq,157,0.0825
7,operating_margin,151,0.0793
8,net_margin,151,0.0793
9,revenue,151,0.0793


In [49]:
# Financial Features Consolidation

def consolidate_financial_features(financial_df: pd.DataFrame) -> pd.DataFrame:
    """
    Consolidate financial features to one row per ticker/reference_date.

    SEC XBRL data can contain multiple fiscal contexts for the same reference date.
    This function keeps the most complete and most useful row per ticker/date.
    """

    df = financial_df.copy()

    df["ticker"] = df["ticker"].str.upper().str.strip()
    df["reference_date"] = pd.to_datetime(df["reference_date"], errors="coerce")
    df["fiscal_period"] = df["fiscal_period"].astype(str).str.upper().str.strip()

    # Prefer quarterly records over FY records when multiple rows exist for the same date
    preferred_periods = ["Q1", "Q2", "Q3", "Q4"]

    df["period_priority"] = np.where(
        df["fiscal_period"].isin(preferred_periods),
        1,
        0,
    )

    # Count how complete each row is
    ignore_cols = ["ticker", "reference_date", "fiscal_year", "fiscal_period"]

    feature_cols = [
        col for col in df.columns
        if col not in ignore_cols
    ]

    df["non_null_count"] = df[feature_cols].notna().sum(axis=1)

    # Sort best rows first
    df = df.sort_values(
        ["ticker", "reference_date", "period_priority", "non_null_count", "fiscal_year"],
        ascending=[True, True, False, False, False],
    )

    consolidated_df = df.drop_duplicates(
        subset=["ticker", "reference_date"],
        keep="first",
    ).copy()

    consolidated_df = consolidated_df.drop(
        columns=["period_priority", "non_null_count"],
        errors="ignore",
    )

    consolidated_df = consolidated_df.sort_values(
        ["ticker", "reference_date"]
    ).reset_index(drop=True)

    return consolidated_df

In [50]:
financial_features_consolidated_df = consolidate_financial_features(financial_features_df)

print("Before consolidation:", financial_features_df.shape)
print("After consolidation:", financial_features_consolidated_df.shape)

display(financial_features_consolidated_df.head())

Before consolidation: (1904, 24)
After consolidation: (451, 24)


,ticker,reference_date,fiscal_year,fiscal_period,assets,cash_and_equivalents,liabilities,net_income,operating_income,rd_expense,revenue,stockholders_equity,operating_margin,net_margin,liabilities_to_assets,cash_to_assets,equity_to_assets,rd_to_revenue,revenue_growth_qoq,revenue_growth_yoy,net_income_growth_qoq,net_income_growth_yoy,assets_growth_qoq,liabilities_growth_qoq
0,AAPL,2015-03-28,"2,016.0000",Q2,"261,194,000,000.0000","14,489,000,000.0000","132,188,000,000.0000","31,593,000,000.0000","42,524,000,000.0000","3,813,000,000.0000",NaN,"129,006,000,000.0000",NaN,NaN,0.5061,0.0555,0.4939,NaN,NaN,NaN,0.7528,0.7528,-0.0027,-0.0460
1,AAPL,2015-06-27,"2,016.0000",Q3,"273,151,000,000.0000","15,319,000,000.0000","147,474,000,000.0000","42,270,000,000.0000","56,607,000,000.0000","5,847,000,000.0000",NaN,"125,677,000,000.0000",NaN,NaN,0.5399,0.0561,0.4601,NaN,NaN,NaN,2.9590,2.1152,0.0000,0.0000
2,AAPL,2015-09-26,"2,017.0000",Q2,"290,479,000,000.0000","21,120,000,000.0000","171,124,000,000.0000","53,394,000,000.0000","71,230,000,000.0000","8,067,000,000.0000",NaN,"119,355,000,000.0000",NaN,NaN,0.5891,0.0727,0.4109,NaN,NaN,NaN,0.0000,0.2632,0.0000,0.0000
3,AAPL,2015-12-26,"2,017.0000",Q1,"293,284,000,000.0000","16,689,000,000.0000","165,017,000,000.0000","18,361,000,000.0000","24,171,000,000.0000","2,404,000,000.0000",NaN,"128,267,000,000.0000",NaN,NaN,0.5627,0.0569,0.4373,NaN,NaN,NaN,0.0000,-0.6561,0.0000,0.0000
4,AAPL,2016-03-26,"2,017.0000",Q2,"305,277,000,000.0000","21,514,000,000.0000","174,820,000,000.0000","28,877,000,000.0000","38,158,000,000.0000","4,915,000,000.0000",NaN,"130,457,000,000.0000",NaN,NaN,0.5727,0.0705,0.4273,NaN,NaN,NaN,1.7460,0.5727,0.0000,0.0000


In [51]:
# Validating duplicates

duplicate_financial_dates = financial_features_consolidated_df.duplicated(
    subset=["ticker", "reference_date"]
).sum()

print("Duplicated ticker/reference_date rows:", duplicate_financial_dates)

Duplicated ticker/reference_date rows: 0


In [52]:
# Validating missing values

missing_values = financial_features_consolidated_df.isna().sum()
missing_pct = financial_features_consolidated_df.isna().mean()

financial_consolidated_missing_summary = pd.DataFrame({
    "column": missing_values.index,
    "missing_values": missing_values.values,
    "missing_pct": missing_pct.values,
})

financial_consolidated_missing_summary = financial_consolidated_missing_summary.sort_values(
    "missing_pct",
    ascending=False
).reset_index(drop=True)

financial_consolidated_missing_summary

,column,missing_values,missing_pct
0,liabilities,183,0.4058
1,liabilities_to_assets,183,0.4058
2,liabilities_growth_qoq,183,0.4058
3,rd_to_revenue,66,0.1463
4,rd_expense,45,0.0998
5,revenue_growth_yoy,34,0.0754
6,net_margin,29,0.0643
7,operating_margin,29,0.0643
8,revenue_growth_qoq,29,0.0643
9,revenue,29,0.0643


In [53]:
# Drop high-missing financial feature columns

high_missing_financial_columns = [
    "liabilities",
    "liabilities_to_assets",
    "liabilities_growth_qoq",
]

financial_features_selected_df = financial_features_consolidated_df.drop(
    columns=high_missing_financial_columns,
    errors="ignore",
).copy()

print("Original consolidated shape:", financial_features_consolidated_df.shape)
print("Selected financial features shape:", financial_features_selected_df.shape)

financial_features_selected_df.head()

Original consolidated shape: (451, 24)
Selected financial features shape: (451, 21)


,ticker,reference_date,fiscal_year,fiscal_period,assets,cash_and_equivalents,net_income,operating_income,rd_expense,revenue,stockholders_equity,operating_margin,net_margin,cash_to_assets,equity_to_assets,rd_to_revenue,revenue_growth_qoq,revenue_growth_yoy,net_income_growth_qoq,net_income_growth_yoy,assets_growth_qoq
0,AAPL,2015-03-28,"2,016.0000",Q2,"261,194,000,000.0000","14,489,000,000.0000","31,593,000,000.0000","42,524,000,000.0000","3,813,000,000.0000",NaN,"129,006,000,000.0000",NaN,NaN,0.0555,0.4939,NaN,NaN,NaN,0.7528,0.7528,-0.0027
1,AAPL,2015-06-27,"2,016.0000",Q3,"273,151,000,000.0000","15,319,000,000.0000","42,270,000,000.0000","56,607,000,000.0000","5,847,000,000.0000",NaN,"125,677,000,000.0000",NaN,NaN,0.0561,0.4601,NaN,NaN,NaN,2.9590,2.1152,0.0000
2,AAPL,2015-09-26,"2,017.0000",Q2,"290,479,000,000.0000","21,120,000,000.0000","53,394,000,000.0000","71,230,000,000.0000","8,067,000,000.0000",NaN,"119,355,000,000.0000",NaN,NaN,0.0727,0.4109,NaN,NaN,NaN,0.0000,0.2632,0.0000
3,AAPL,2015-12-26,"2,017.0000",Q1,"293,284,000,000.0000","16,689,000,000.0000","18,361,000,000.0000","24,171,000,000.0000","2,404,000,000.0000",NaN,"128,267,000,000.0000",NaN,NaN,0.0569,0.4373,NaN,NaN,NaN,0.0000,-0.6561,0.0000
4,AAPL,2016-03-26,"2,017.0000",Q2,"305,277,000,000.0000","21,514,000,000.0000","28,877,000,000.0000","38,158,000,000.0000","4,915,000,000.0000",NaN,"130,457,000,000.0000",NaN,NaN,0.0705,0.4273,NaN,NaN,NaN,1.7460,0.5727,0.0000


In [54]:
# Checking NA again

selected_missing_values = financial_features_selected_df.isna().sum()
selected_missing_pct = financial_features_selected_df.isna().mean()

selected_financial_missing_summary = pd.DataFrame({
    "column": selected_missing_values.index,
    "missing_values": selected_missing_values.values,
    "missing_pct": selected_missing_pct.values,
}).sort_values("missing_pct", ascending=False).reset_index(drop=True)

selected_financial_missing_summary

,column,missing_values,missing_pct
0,rd_to_revenue,66,0.1463
1,rd_expense,45,0.0998
2,revenue_growth_yoy,34,0.0754
3,net_margin,29,0.0643
4,operating_margin,29,0.0643
5,revenue,29,0.0643
6,revenue_growth_qoq,29,0.0643
7,assets,0,0.0000
8,reference_date,0,0.0000
9,ticker,0,0.0000


In [55]:
# Sabing the new data into the data folder

financial_features_selected_df.to_csv(
    "data/financial_features_selected.csv",
    index=False,
)

print("Selected financial features saved successfully.")

Selected financial features saved successfully.


In [56]:
# Model-Ready Dataset Merge

def build_model_ready_dataset(
    market_df: pd.DataFrame,
    filing_df: pd.DataFrame,
    financial_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build the model-ready dataset by combining:
    - daily market features
    - daily filing activity features
    - latest available financial features using as-of merge

    Final granularity:
        one row per ticker/date
    """

    market = market_df.copy()
    filing = filing_df.copy()
    financial = financial_df.copy()

    # Standardize keys
    market["ticker"] = market["ticker"].str.upper().str.strip()
    filing["ticker"] = filing["ticker"].str.upper().str.strip()
    financial["ticker"] = financial["ticker"].str.upper().str.strip()

    market["date"] = pd.to_datetime(market["date"], errors="coerce")
    filing["date"] = pd.to_datetime(filing["date"], errors="coerce")
    financial["reference_date"] = pd.to_datetime(
        financial["reference_date"],
        errors="coerce",
    )

    # 1. Merge market + filing features by ticker/date
    model_base = market.merge(
        filing,
        on=["ticker", "date"],
        how="left",
        validate="one_to_one",
    )

    # Filing features should be zero when no filing activity exists
    filing_feature_cols = [
        "filing_count",
        "form_10k_count",
        "form_10q_count",
        "form_8k_count",
        "filing_count_30d",
        "filing_count_90d",
        "form_8k_count_30d",
        "form_10q_count_180d",
        "form_10k_count_365d",
        "days_since_last_filing",
        "days_since_last_8k",
    ]

    for col in filing_feature_cols:
        if col in model_base.columns:
            model_base[col] = model_base[col].fillna(0)

    # 2. Merge latest available financial features by ticker/date
    merged_frames = []

    for ticker, market_group in model_base.groupby("ticker"):
        market_group = market_group.copy().sort_values("date")

        financial_group = financial[
            financial["ticker"] == ticker
        ].copy().sort_values("reference_date")

        if financial_group.empty:
            merged_frames.append(market_group)
            continue

        merged_ticker = pd.merge_asof(
            market_group,
            financial_group,
            left_on="date",
            right_on="reference_date",
            by="ticker",
            direction="backward",
        )

        merged_frames.append(merged_ticker)

    model_ready_df = pd.concat(merged_frames, ignore_index=True)

    model_ready_df = model_ready_df.sort_values(
        ["ticker", "date"]
    ).reset_index(drop=True)

    return model_ready_df

In [57]:
# Running the merge

model_ready_df = build_model_ready_dataset(
    market_df=market_features_df,
    filing_df=filing_features_df,
    financial_df=financial_features_selected_df,
)

print("model_ready_df shape:", model_ready_df.shape)

display(model_ready_df.head())

model_ready_df shape: (27730, 57)


,ticker,date,open,high,low,close,volume,source,collected_at,daily_return,log_return,price_change_7d,price_change_30d,price_change_90d,volatility_7d,volatility_30d,volatility_90d,avg_volume_30d,std_volume_30d,volume_change_30d,volume_zscore_30d,avg_return_30d,std_return_30d,return_zscore_30d,daily_range,open_gap,filing_count,form_10k_count,form_10q_count,form_8k_count,filing_count_30d,filing_count_90d,form_8k_count_30d,form_10q_count_180d,form_10k_count_365d,days_since_last_filing,days_since_last_8k,reference_date,fiscal_year,fiscal_period,assets,cash_and_equivalents,net_income,operating_income,rd_expense,revenue,stockholders_equity,operating_margin,net_margin,cash_to_assets,equity_to_assets,rd_to_revenue,revenue_growth_qoq,revenue_growth_yoy,net_income_growth_qoq,net_income_growth_yoy,assets_growth_qoq
0,AAPL,2015-05-13,28.1670,28.4023,28.1062,28.1376,"155,379,331.0000",stooq,2026-05-21 19:51:02.287737+00:00,0.0012,0.0012,-0.0167,0.0170,0.1620,0.0130,0.0132,0.0161,"217,353,230.9000","94,016,158.6912",-0.2851,-0.6592,0.0006,0.0132,0.0411,0.0105,0.0022,1,0,0,1,1.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,2015-03-28,"2,016.0000",Q2,"261,194,000,000.0000","14,489,000,000.0000","31,593,000,000.0000","42,524,000,000.0000","3,813,000,000.0000",NaN,"129,006,000,000.0000",NaN,NaN,0.0555,0.4939,NaN,NaN,NaN,0.7528,0.7528,-0.0027
1,AAPL,2015-05-14,28.4505,28.7970,28.3955,28.7970,"202,444,982.0000",stooq,2026-05-21 19:51:02.287737+00:00,0.0234,0.0232,0.0293,0.0423,0.2237,0.0128,0.0138,0.0159,"218,011,922.3333","93,833,933.5375",-0.0714,-0.1659,0.0015,0.0138,1.5892,0.0139,0.0111,0,0,0,0,1.0000,1.0000,1.0000,0.0000,0.0000,1.0000,1.0000,2015-03-28,"2,016.0000",Q2,"261,194,000,000.0000","14,489,000,000.0000","31,593,000,000.0000","42,524,000,000.0000","3,813,000,000.0000",NaN,"129,006,000,000.0000",NaN,NaN,0.0555,0.4939,NaN,NaN,NaN,0.7528,0.7528,-0.0027
2,AAPL,2015-05-15,28.8244,28.9237,28.6280,28.7557,"171,052,227.0000",stooq,2026-05-21 19:51:02.287737+00:00,-0.0014,-0.0014,0.0345,0.0320,0.2218,0.0122,0.0138,0.0159,"218,883,610.9333","93,250,996.0012",-0.2185,-0.5129,0.0011,0.0138,-0.1871,0.0103,0.0010,0,0,0,0,1.0000,1.0000,1.0000,0.0000,0.0000,2.0000,2.0000,2015-03-28,"2,016.0000",Q2,"261,194,000,000.0000","14,489,000,000.0000","31,593,000,000.0000","42,524,000,000.0000","3,813,000,000.0000",NaN,"129,006,000,000.0000",NaN,NaN,0.0555,0.4939,NaN,NaN,NaN,0.7528,0.7528,-0.0027
3,AAPL,2015-05-18,28.6644,29.1845,28.6596,29.0639,"227,617,764.0000",stooq,2026-05-21 19:51:02.287737+00:00,0.0107,0.0107,0.0390,0.0265,0.2176,0.0124,0.0136,0.0159,"220,895,204.6667","92,748,755.8727",0.0304,0.0725,0.0010,0.0136,0.7179,0.0181,-0.0032,0,0,0,0,1.0000,1.0000,1.0000,0.0000,0.0000,5.0000,5.0000,2015-03-28,"2,016.0000",Q2,"261,194,000,000.0000","14,489,000,000.0000","31,593,000,000.0000","42,524,000,000.0000","3,813,000,000.0000",NaN,"129,006,000,000.0000",NaN,NaN,0.0555,0.4939,NaN,NaN,NaN,0.7528,0.7528,-0.0027
4,AAPL,2015-05-19,29.1776,29.2201,28.9530,29.0373,"199,891,497.0000",stooq,2026-05-21 19:51:02.287737+00:00,-0.0009,-0.0009,0.0190,0.0364,0.1716,0.0111,0.0134,0.0154,"222,312,705.3667","92,066,804.2472",-0.1009,-0.2435,0.0013,0.0134,-0.1633,0.0092,0.0039,0,0,0,0,1.0000,1.0000,1.0000,0.0000,0.0000,6.0000,6.0000,2015-03-28,"2,016.0000",Q2,"261,194,000,000.0000","14,489,000,000.0000","31,593,000,000.0000","42,524,000,000.0000","3,813,000,000.0000",NaN,"129,006,000,000.0000",NaN,NaN,0.0555,0.4939,NaN,NaN,NaN,0.7528,0.7528,-0.0027


In [58]:
# Verifying the granuliarity

duplicates = model_ready_df.duplicated(subset=["ticker", "date"]).sum()

print("Duplicated ticker/date rows:", duplicates)
print("Rows:", len(model_ready_df))
print("Tickers:", model_ready_df["ticker"].nunique())
print("Date range:", model_ready_df["date"].min(), "→", model_ready_df["date"].max())

Duplicated ticker/date rows: 0
Rows: 27730
Tickers: 10
Date range: 2015-05-13 00:00:00 → 2026-05-21 00:00:00


In [59]:
# Veriifying missing values in the model data

model_missing_values = model_ready_df.isna().sum()
model_missing_pct = model_ready_df.isna().mean()

model_missing_summary = pd.DataFrame({
    "column": model_missing_values.index,
    "missing_values": model_missing_values.values,
    "missing_pct": model_missing_pct.values,
}).sort_values("missing_pct", ascending=False).reset_index(drop=True)

model_missing_summary.head(40)

,column,missing_values,missing_pct
0,rd_to_revenue,4009,0.1446
1,rd_expense,2773,0.1000
2,revenue_growth_yoy,2030,0.0732
3,net_margin,1711,0.0617
4,revenue_growth_qoq,1711,0.0617
5,revenue,1711,0.0617
6,operating_margin,1711,0.0617
7,date,0,0.0000
8,ticker,0,0.0000
9,daily_return,0,0.0000


In [60]:
# Finan dataframe for the model

model_ready_df["has_missing_financial_data"] = model_ready_df[
    [
        "rd_to_revenue",
        "rd_expense",
        "revenue_growth_yoy",
        "net_margin",
        "revenue_growth_qoq",
        "revenue",
        "operating_margin",
    ]
].isna().any(axis=1).astype(int)

print(model_ready_df["has_missing_financial_data"].value_counts(normalize=True))

has_missing_financial_data
0   0.8439
1   0.1561
Name: proportion, dtype: float64


In [61]:
# Saving the dataset to disk

model_ready_df.to_csv("data/model_ready_dataset.csv", index=False)

print("Model-ready dataset saved successfully.")
print("Shape:", model_ready_df.shape)

Model-ready dataset saved successfully.
Shape: (27730, 58)


## Model-Ready Dataset — Missing Values Decision

The final `model_ready_df` was created by combining market features, SEC filing activity features, and the latest available financial features for each company/date.

The remaining missing values are concentrated only in financial variables such as R&D intensity, revenue, revenue growth, and margin-related features. This is expected because SEC XBRL concepts are not reported uniformly across all companies and all periods.

No missing values were found in the market feature layer or filing activity feature layer.

The remaining financial missing values will not be imputed in this notebook. Instead, they will be handled later inside the machine learning pipeline using a proper imputation strategy, such as median imputation, after train/test splitting. This avoids introducing data leakage during model training.

A helper flag, `has_missing_financial_data`, was added to indicate rows where at least one selected financial feature is missing.

The final dataset is now ready for the anomaly detection modeling stage.